In [ ]:
import pandas as pd
import requests
import os
import time
from pathlib import Path
import logging
from typing import Set, Tuple, Dict, Any, Optional
import sys
import random
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import concurrent.futures
from tqdm import tqdm

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('wowy_scraper.log'),
        logging.StreamHandler()
    ]
)

class RateLimiter:
    """Simple rate limiter to respect API limits"""
    def __init__(self, min_time: float = 1.0, max_time: float = 3.0, backoff_factor: float = 1.5):
        self.min_time = min_time
        self.max_time = max_time
        self.backoff_factor = backoff_factor
        self.last_request_time = 0
        self.current_wait = min_time
        
    def wait(self) -> None:
        """Wait the appropriate amount of time between requests"""
        now = time.time()
        elapsed = now - self.last_request_time
        
        # If we need to wait more
        if elapsed < self.current_wait:
            time_to_sleep = self.current_wait - elapsed
            time.sleep(time_to_sleep)
            
        # Update the last request time
        self.last_request_time = time.time()
    
    def success(self) -> None:
        """Call when a request succeeds to potentially reduce wait time"""
        self.current_wait = max(self.min_time, self.current_wait / self.backoff_factor)
        
    def failure(self) -> None:
        """Call when a request fails to increase wait time"""
        self.current_wait = min(self.max_time, self.current_wait * self.backoff_factor)


def create_session() -> requests.Session:
    """Create a requests session with retry logic"""
    session = requests.Session()
    
    # Define retry strategy
    retry_strategy = Retry(
        total=5,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"]
    )
    
    # Mount the adapter to the session
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    
    # Set headers
    session.headers.update({
        "Host": "stats.nba.com",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:72.0) Gecko/20100101 Firefox/72.0",
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "en-US,en;q=0.5",
        "Accept-Encoding": "gzip, deflate, br",
        "Connection": "keep-alive",
        "Referer": "https://stats.nba.com/"
    })
    
    return session


def wowy_shift(
    session: requests.Session,
    rate_limiter: RateLimiter,
    team_id: str,
    player_id: str,
    seasons: list,
    ps: bool = False
) -> pd.DataFrame:
    """Get WOWY stats for a player with improved error handling and rate limiting"""
    
    if ps == False:
        s_type = 'Regular Season'
    elif ps == 'all':
        s_type = 'All'
    else:
        s_type = 'Playoffs'
    
    # Prepare URL and parameters
    wowy_url = "https://api.pbpstats.com/get-wowy-stats/nba"
    
    # Get player on stats
    wowy_params_on = {
        "0Exactly1OnFloor": player_id,
        "TeamId": team_id,
        "Season": ",".join(seasons),
        "SeasonType": s_type,
        "Type": "Player",
    }
    
    # Wait according to rate limiter
    rate_limiter.wait()
    
    try:
        wowy_response = session.get(wowy_url, params=wowy_params_on, timeout=30)
        wowy_response.raise_for_status()
        
        wowy = wowy_response.json()
        player_stats_on = wowy.get("multi_row_table_data", [])
        
        # Signal success to rate limiter
        rate_limiter.success()
    except Exception as e:
        # Signal failure to rate limiter
        rate_limiter.failure()
        logging.error(f"Error fetching player ON stats - {player_id} - {team_id}: {e}")
        # Re-raise to be caught by the caller
        raise
    
    # Get player off stats
    wowy_params_off = {
        "0Exactly0OnFloor": player_id,
        "TeamId": team_id,
        "Season": ",".join(seasons),
        "SeasonType": s_type,
        "Type": "Player",
    }
    
    # Wait according to rate limiter
    rate_limiter.wait()
    
    try:
        wowy_response = session.get(wowy_url, params=wowy_params_off, timeout=30)
        wowy_response.raise_for_status()
        
        wowy = wowy_response.json()
        player_stats_off = wowy.get("multi_row_table_data", [])
        
        # Signal success to rate limiter
        rate_limiter.success()
    except Exception as e:
        # Signal failure to rate limiter
        rate_limiter.failure()
        logging.error(f"Error fetching player OFF stats - {player_id} - {team_id}: {e}")
        # Re-raise to be caught by the caller
        raise
    
    # Create DataFrames
    df_on = pd.DataFrame(player_stats_on) if player_stats_on else pd.DataFrame()
    if not df_on.empty:
        df_on['on'] = True
        
    df_off = pd.DataFrame(player_stats_off) if player_stats_off else pd.DataFrame()
    if not df_off.empty:
        df_off['on'] = False
    
    # Combine dataframes
    if df_on.empty and df_off.empty:
        # Return empty DataFrame with expected columns
        return pd.DataFrame(columns=['TeamId', 'on'])
    elif df_on.empty:
        return df_off
    elif df_off.empty:
        return df_on
    else:
        return pd.concat([df_on, df_off])


def setup_folders(base_year: int, end_year: int, ps: bool = False) -> None:
    """Create folders for each season if they don't exist."""
    trail = 'ps' if ps else ''
    for year in range(base_year, end_year + 1):
        Path(f"data/{year}{trail}").mkdir(parents=True, exist_ok=True)


def load_processed_data(year: int, ps: bool = False) -> Dict[str, pd.DataFrame]:
    """Load all processed data for a given year into memory for faster access."""
    trail = 'ps' if ps else ''
    year_dir = Path(f"data/{year}{trail}")
    processed_data = {}
    
    if year_dir.exists():
        for file in year_dir.glob("*.csv"):
            try:
                nba_id = file.stem
                df = pd.read_csv(file)
                processed_data[nba_id] = df
            except Exception as e:
                logging.error(f"Error reading file {file}: {e}")
    
    return processed_data


def get_processed_combinations(processed_data: Dict[str, pd.DataFrame]) -> Set[Tuple[str, str]]:
    """Get already processed player-team combinations from loaded data."""
    processed = set()
    for nba_id, df in processed_data.items():
        if not df.empty and 'TeamId' in df.columns:
            team_ids = df['TeamId'].unique()
            for team_id in team_ids:
                processed.add((nba_id, str(team_id)))
    return processed


def process_player_team(
    args: Dict[str, Any]
) -> Optional[Tuple[str, str, pd.DataFrame]]:
    """Process a single player-team combination."""
    nba_id = args['nba_id']
    team_id = args['team_id']
    year = args['year']
    is_postseason = args['is_postseason']
    seasons = args['seasons']
    
    session = create_session()
    rate_limiter = RateLimiter(min_time=1.0, max_time=5.0)
    
    try:
        logging.debug(f"Processing {nba_id} - {team_id} for {year}")
        
        result = wowy_shift(
            session=session,
            rate_limiter=rate_limiter,
            team_id=team_id,
            player_id=str(int(nba_id)),
            seasons=seasons,
            ps=is_postseason
        )
        
        return (str(nba_id), str(team_id), result)
        
    except Exception as e:
        logging.error(f"Error processing {nba_id} - {team_id} for {year}: {e}")
        return None


def process_season_data(
    year: int, 
    is_postseason: bool, 
    index_df: pd.DataFrame,
    max_workers: int = 4
) -> None:
    """Process data for a single season with concurrent processing."""
    trail = 'ps' if is_postseason else ''
    season_start = str(year - 1)
    season_end = str(year)
    seasons = [f"{season_start}-{season_end[-2:]}"]
    
    # Ensure nba_id is integer type
    index_df['nba_id'] = index_df['nba_id'].astype(int)
    
    # Load existing data
    processed_data = load_processed_data(year, is_postseason)
    processed_combinations = get_processed_combinations(processed_data)
    
    # Filter to players for this year
    year_data = index_df[index_df['year'] == year]
    
    # Create task list
    tasks = []
    for _, row in year_data.iterrows():
        nba_id = row['nba_id']
        team_id = row['team_id']
        
        # Skip if already processed
        if (str(nba_id), str(team_id)) in processed_combinations:
            continue
            
        tasks.append({
            'nba_id': nba_id,
            'team_id': team_id,
            'year': year,
            'is_postseason': is_postseason,
            'seasons': seasons
        })
    
    logging.info(f"Found {len(tasks)} tasks to process for {year} {'playoffs' if is_postseason else 'regular season'}")
    
    # If no tasks, return early
    if not tasks:
        logging.info(f"No new data to process for {year}")
        return
    
    # Process tasks concurrently
    results = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(process_player_team, task): task for task in tasks}
        
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures)):
            task = futures[future]
            try:
                result = future.result()
                if result:
                    results.append(result)
            except Exception as e:
                nba_id = task['nba_id']
                team_id = task['team_id']
                logging.error(f"Task failed for {nba_id} - {team_id}: {e}")
    
    # Save results
    for nba_id, team_id, result_df in results:
        if result_df.empty:
            continue
            
        output_file = Path(f"data/{int(year)}{trail}/{int(nba_id)}.csv")
        
        try:
            # If file exists in our loaded data
            if nba_id in processed_data:
                existing_data = processed_data[nba_id]
                combined_data = pd.concat([existing_data, result_df], ignore_index=True)
                combined_data.drop_duplicates().to_csv(output_file, index=False)
                # Update our in-memory data
                processed_data[nba_id] = combined_data
            else:
                result_df.to_csv(output_file, index=False)
                # Add to our in-memory data
                processed_data[nba_id] = result_df
        except Exception as e:
            logging.error(f"Error saving data for {nba_id}: {e}")


def main():
    # Set maximum number of concurrent workers
    # Adjust this number based on your connection and the API's rate limits
    MAX_WORKERS = 3
    
    # Load data
    try:
        logging.info("Loading index data...")
        index_reg = pd.read_csv('https://raw.githubusercontent.com/gabriel1200/site_Data/refs/heads/master/index_master.csv')
        index_reg.dropna(subset=['nba_id', 'team_id'], inplace=True)
        index_reg = index_reg[index_reg.team != 'TOT']
        
        index_ps = pd.read_csv('https://raw.githubusercontent.com/gabriel1200/site_Data/refs/heads/master/index_master_ps.csv')
        index_ps.dropna(subset=['nba_id', 'team_id'], inplace=True)
        index_ps = index_ps[index_ps.team != 'TOT']
        
        logging.info(f"Loaded {len(index_reg)} regular season records and {len(index_ps)} playoff records")
    except Exception as e:
        logging.error(f"Error loading index files: {e}")
        return

    # Create folders
    setup_folders(2025, 2026)
    #setup_folders(2025, 2025, ps=True)

    # Process regular season first (since there's likely more data)
    for year in range(2025, 2026):
        logging.info(f"Processing regular season {year}")
        process_season_data(year, False, index_reg, max_workers=MAX_WORKERS)
'''rocess postseason
    for year in range(2025, 2026):
        logging.info(f"Processing postseason {year}")
        process_season_data(year, True, index_ps, max_workers=MAX_WORKERS)
'''


if __name__ == "__main__":
    main()